In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:

# =========================
# 1. IMPORTS
# =========================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import re

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

warnings.filterwarnings('ignore')


# =========================
# 2. LOAD DATA
# =========================
df_train = pd.read_csv('/kaggle/input/competitions/nlp-getting-started/train.csv')
df_test = pd.read_csv('/kaggle/input/competitions/nlp-getting-started/test.csv')


# =========================
# 3. BASIC INFO
# =========================
print(df_train.shape)
print(df_train.head())
print(df_train.isnull().sum())


# =========================
# 4. VISUALIZATION
# =========================
sns.countplot(x='target', data=df_train)
plt.title("Target Distribution")
plt.show()


# =========================
# 5. TEXT CLEANING
# =========================
def clean_text(text):
    text = str(text)
    text = re.sub(r'http\S+', '', text)
    text = re.sub(r'[^a-zA-Z]', ' ', text)
    text = text.lower()
    return text

df_train['clean_text'] = df_train['text'].apply(clean_text)
df_test['clean_text'] = df_test['text'].apply(clean_text)


# =========================
# 6. FEATURE EXTRACTION
# =========================
vectorizer = TfidfVectorizer(max_features=5000)

X = vectorizer.fit_transform(df_train['clean_text'])
y = df_train['target']


# =========================
# 7. TRAIN TEST SPLIT
# =========================
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42
)


# =========================
# 8. MODEL TRAINING
# =========================
model = LogisticRegression()
model.fit(X_train, y_train)


# =========================
# 9. EVALUATION
# =========================
y_pred = model.predict(X_val)
print("Accuracy:", accuracy_score(y_val, y_pred))


# =========================
# 10. PREDICTION
# =========================
X_test = vectorizer.transform(df_test['clean_text'])
predictions = model.predict(X_test)


# =========================
# 11. SUBMISSION FILE
# =========================
submission = pd.DataFrame({
    'id': df_test['id'],
    'target': predictions
})

submission.to_csv('submission.csv', index=False)
print("Submission file created!")


# =========================
# 12. SAFE PLOT SAVE (optional)
# =========================
plt.savefig('plot.png', dpi=300, bbox_inches='tight')
plt.close('all')